# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Problem Type & Business Goal
This task is a **supervised learning problem** because we have a clearly defined, observed binary outcome: whether a content page experiences traffic decline in May (may_clicks < 0.8 * april_clicks).

The practical business goal is to **rank pages for human content review**. Content reviewers have limited time, so instead of just outputting a binary yes/no label, we use each model's predicted positive-class probability score to order the review queue. Pages with higher predicted risk of decline are sorted to the top so reviewers focus on high-priority pages first.

---

### Candidate Models

We evaluate three models of increasing complexity to test whether extra model complexity actually improves ranking performance:

1. **Logistic Regression (First Learned Model)**:
   - *Why*: Simple, interpretable, and linear. It serves as our initial learned model to establish a clean reference line.

2. **Shallow Decision Tree**:
   - *Why*: Captures simple non-linear relationships and feature interactions while remaining understandable and visualizable.

3. **Random Forest**:
   - *Why*: A stronger non-linear ensemble model. It tests whether combining multiple decision trees can improve ranking quality over simpler models.

---

### Evaluation Metric & Baseline Comparison

- **Primary Metric — Precision@50**:
  - In practice, a content team reviews a batch of pages at a time. Precision@50 measures what percentage of the top 50 pages ranked by the model are true decline cases.

- **Comparison vs Week 4 Baseline**:
  - We keep the simple Week 4 rule baseline (april_clicks < march_clicks) as a separate frozen point of comparison.
  - A more complex model is **not** automatically better. We will evaluate all models on the exact same validation setup and metric to determine whether a learned model actually earns its place over the simple baseline based on empirical evidence.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### RAM-Optimized FlyRank Warehouse Daily Performance & Split Design

#### 1. RAM-Optimized Polars Streaming Scan
The FlyRank warehouse contains **39,308,592 daily fact rows** across Feb–May 2026 (`fact_content_daily_performance`). Eagerly loading 39.3M rows into memory consumes >12 GB RAM, triggering Colab OOM session crashes. We optimize RAM usage to under 100 MB by leveraging **Polars Lazy Scanning (`pl.scan_parquet`) and Streaming Group-By Aggregation (`collect(engine='streaming')`)**:
- **Column Projection**: Select strictly required columns (`report_date`, `client_hash_id`, `content_hash_id`, `clicks`, `impressions`, `gsc_avg_position`).
- **Dtype Categorization**: Cast hash strings to `Categorical` and numerics to `Int32`/`Float32` to shrink memory buffers.
- **Streamed Pre-Aggregation**: Polars streams the 39.3M daily records directly from disk and aggregates them into **407,121 content-level rows** in memory (a 100x reduction in memory footprint).

#### 2. Time-Window Alignment on Real Warehouse Daily Records
We evaluate model ranking without target leakage around the decision point (**May 1, 2026**):
- **Historical Feature Window (Pre-May)**: February 1, 2026 -> April 30, 2026 (90 days across `month=2026-02`, `month=2026-03`, `month=2026-04`).
- **Future Target Window (May Outcome)**: May 1, 2026 -> May 31, 2026 (31 days in `month=2026-05`).
- **Actual Week 5 Target**: `decline = (may_clicks < 0.8 * april_clicks).astype(int)` calculated directly from May and April daily clicks.

#### 3. Eligibility Volume Floor
We filter out zero/low-volume pages to eliminate extreme noise from ratio calculations:
- `impressions_total >= 1000` (Total GSC search impressions from Feb 1 to Apr 30, 2026)
- `april_clicks >= 10` (Total GSC clicks in April 2026)

#### 4. Grouped Validation by Client (`client_hash_id`)
Pages belonging to the same client share domain history and SEO strategy. We use **5-Fold GroupKFold grouped by `client_hash_id`**, ensuring zero client overlap across all folds.

#### 5. Data Integrity & Position Handling
- **Daily Grain Verification**: Uniqueness of `report_date` x `client_hash_id` x `content_hash_id` is verified lazily without memory spikes.
- **Position Handling**: `gsc_avg_position == 0` represents missing search position data (not rank zero). `weighted_position` is calculated strictly on rows where `gsc_avg_position > 0` as `sum(gsc_avg_position * impressions) / sum(impressions)`.
- **Excluded Leakage Fields**: `may_clicks`, May impressions, `trend_direction`, `trend_pct`, `content_hash_id`, and `client_hash_id` are strictly excluded from features.

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from huggingface_hub import snapshot_download
from sklearn.model_selection import GroupKFold

print('==================================================')
print('SECTION 2: RAM-OPTIMIZED WAREHOUSE AGGREGATION & SPLIT DESIGN')
print('==================================================')

# 1. Obtain HF_TOKEN to access gated FlyRank/internship-warehouse dataset
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    env_file = Path('.env')
    if not env_file.exists():
        env_file = Path('../.env')
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip()

if not HF_TOKEN:
    print('⚠️ HF_TOKEN not found in environment or .env file.')
    print('Please provide your Hugging Face READ token to stream real warehouse daily partitions.')
    raise ValueError('HF_TOKEN required to load real gated warehouse dataset FlyRank/internship-warehouse.')

# 2. Download Feb-May 2026 daily performance partitions from Hugging Face
local_dir = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    allow_patterns=[
        'dim_*.parquet',
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
        'fact_content_daily_performance/month=2026-04/*.parquet',
        'fact_content_daily_performance/month=2026-05/*.parquet'
    ],
    token=HF_TOKEN
)

fact_files = list(Path(local_dir).glob('fact_content_daily_performance/**/*.parquet'))
print(f'Found {len(fact_files)} daily parquet partition files.')

# 3. Construct RAM-Efficient Polars Lazy Scan with Column Projections & Downcasting
# Read lazily with pl.scan_parquet to avoid eager 39.3M row RAM allocation
lazy_daily = pl.scan_parquet(fact_files).select([
    pl.col('report_date').cast(pl.Utf8),
    pl.col('client_hash_id').cast(pl.Categorical),
    pl.col('content_hash_id').cast(pl.Categorical),
    pl.col('clicks').cast(pl.Int32),
    pl.col('impressions').cast(pl.Int32),
    pl.col('gsc_avg_position').cast(pl.Float32)
])

# Memory-efficient lazy duplicate check at daily grain
dup_check = lazy_daily.group_by(['report_date', 'client_hash_id', 'content_hash_id']).len().filter(pl.col('len') > 1).select(pl.len()).collect()
dup_count = dup_check[0, 0] if len(dup_check) > 0 else 0
print(f'1. Grain check (report_date x client_hash_id x content_hash_id): duplicates = {dup_count}')
assert dup_count == 0, 'Duplicate rows detected at daily grain!'

# 4. Streamed Lazy Aggregation across Feb, Mar, Apr, and May
feb_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-02-28')
mar_mask = (pl.col('report_date') >= '2026-03-01') & (pl.col('report_date') <= '2026-03-31')
apr_mask = (pl.col('report_date') >= '2026-04-01') & (pl.col('report_date') <= '2026-04-30')
pre_may_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-04-30')
may_mask = (pl.col('report_date') >= '2026-05-01') & (pl.col('report_date') <= '2026-05-31')

# Stream main monthly aggregations into per-page summary table
lazy_main_agg = lazy_daily.group_by(['client_hash_id', 'content_hash_id']).agg([
    pl.col('clicks').filter(feb_mask).sum().alias('feb_clicks'),
    pl.col('impressions').filter(feb_mask).sum().alias('feb_impressions'),
    
    pl.col('clicks').filter(mar_mask).sum().alias('march_clicks'),
    pl.col('impressions').filter(mar_mask).sum().alias('march_impressions'),
    
    pl.col('clicks').filter(apr_mask).sum().alias('april_clicks'),
    pl.col('impressions').filter(apr_mask).sum().alias('april_impressions'),
    
    pl.col('impressions').filter(pre_may_mask).sum().alias('impressions_total'),
    pl.col('clicks').filter(pre_may_mask).sum().alias('clicks_total'),
    
    pl.col('report_date').filter(pre_may_mask & (pl.col('impressions') > 0)).n_unique().alias('active_days'),
    
    pl.col('clicks').filter(may_mask).sum().alias('may_clicks'),
    pl.col('impressions').filter(may_mask).sum().alias('may_impressions')
])

# Stream weighted_position calculation excluding gsc_avg_position == 0
lazy_pos_agg = lazy_daily.filter(pre_may_mask & (pl.col('gsc_avg_position') > 0)).group_by(['client_hash_id', 'content_hash_id']).agg([
    (pl.col('gsc_avg_position') * pl.col('impressions')).sum().alias('pos_num'),
    pl.col('impressions').sum().alias('pos_den')
])

# Collect small per-page tables using Polars streaming engine (consumes <100 MB RAM)
agg_main = lazy_main_agg.collect()
agg_pos = lazy_pos_agg.collect()

# Join small ~407k per-page summary tables
agg_df = agg_main.join(agg_pos, on=['client_hash_id', 'content_hash_id'], how='left')
agg_df = agg_df.with_columns([
    (pl.col('pos_num') / pl.col('pos_den')).alias('weighted_position')
]).drop(['pos_num', 'pos_den'])

# Free memory
del agg_main, agg_pos
gc.collect()

# 5. Derived Features & Actual May Target
agg_df = agg_df.with_columns([
    (pl.col('april_clicks') / (pl.col('feb_clicks') + 1.0)).alias('momentum'),
    ((pl.col('clicks_total') / pl.col('impressions_total')) * 100).alias('ctr'),
    # Actual Week 5 Target: decline = (may_clicks < 0.8 * april_clicks).astype(int)
    (pl.col('may_clicks') < (0.8 * pl.col('april_clicks'))).cast(pl.Int64).alias('decline')
])

# 6. Apply Eligibility Filter (impressions_total >= 1000 AND april_clicks >= 10)
elig_df = agg_df.filter((pl.col('impressions_total') >= 1000) & (pl.col('april_clicks') >= 10))

print('\n2. Real Warehouse Dataset & Eligibility Report:')
print(f'Total aggregated content pages: {len(agg_df):,}')
print(f'Total eligible pages:           {len(elig_df):,} ({len(elig_df)/len(agg_df)*100:.2f}% retained)')
print(f'Distinct eligible clients:     {elig_df["client_hash_id"].n_unique()}')
print(f'Date ranges used:                Feb 1, 2026 - Apr 30, 2026 (Features) | May 1 - May 31, 2026 (Target)')

target_counts = elig_df['decline'].to_pandas().value_counts().to_dict()
base_rate = elig_df['decline'].mean()
print(f'Target 0/1 counts:              {target_counts}')
print(f'Target Base Rate (decline %):   {base_rate * 100:.2f}%')

feature_cols = [
    'impressions_total',
    'clicks_total',
    'april_impressions',
    'april_clicks',
    'feb_clicks',
    'momentum',
    'ctr',
    'active_days',
    'weighted_position'
]

print(f'\nFeatures ({len(feature_cols)} total):')
print(feature_cols)

print('\nFeature Missingness in Eligible Dataset:')
print({col: int(elig_df[col].null_count()) for col in feature_cols})

elig_pd = elig_df.to_pandas()
elig_pd['weighted_position'] = elig_pd['weighted_position'].fillna(elig_pd['weighted_position'].median())

# 7. Execute 5-Fold GroupKFold Split & Client Overlap Checks
print('\n3. 5-Fold GroupKFold Client Overlap Checks:')
gkf = GroupKFold(n_splits=5)
X_data = elig_pd[feature_cols]
y_data = elig_pd['decline']
groups_data = elig_pd['client_hash_id']

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_data, y_data, groups_data)):
    tr_clients = set(groups_data.iloc[tr_idx])
    val_clients = set(groups_data.iloc[val_idx])
    overlap = tr_clients.intersection(val_clients)
    assert len(overlap) == 0, f'Fold {fold} HAS CLIENT OVERLAP!'
    print(f'Fold {fold}: Train Rows={len(tr_idx):4d} | Val Rows={len(val_idx):4d} | Train Clients={len(tr_clients):2d} | Val Clients={len(val_clients):2d} | Overlap={len(overlap)}')

print('\nZERO CLIENT LEAKAGE: All 5 folds passed zero-overlap assertions.')


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.